# Export `financial_data` to NDJSON

Produces the file-based counterpart of `financial_data` so that a Spark variant
can run without MongoDB in the loop.

`companies` needs no export: the Brreg bulk download `enheter_alle.json`
already is the file, and the profiling pass confirmed the raw file and the
MongoDB collection hold an identical set of 64 top-level fields.
`financial_data` has no file equivalent, because it was assembled from ~1.17M
individual Regnskapsregisteret API calls. **This export is a reconstruction,
not the original ingestion path.** A genuinely file-native pipeline would have
written each API response to disk as it arrived and never involved MongoDB.
That variant would also have hit the small-files problem, since 1.17M
individual responses is pathological for distributed processing. Re-fetching to
demonstrate that is not practical at 1 req/s, so the report should describe
this file as equivalent in content but not in provenance.

NDJSON rather than a single JSON array, for three reasons: Spark's writer emits
only NDJSON, so an array would require collecting 1.17M documents through the
driver; an array is not splittable on read, so the financial side would become
single-threaded like the companies side; and appending one JSON object per line
is what an incremental fetch loop actually produces. The asymmetry against
`enheter_alle.json` is therefore realistic rather than a flaw: a bulk download
and an incremental fetch genuinely differ in framing. **Writes are staged
through container-local disk.** Spark's commit renames fail intermittently
against Dropbox, and they fail after `mode("overwrite")` has deleted the
previous export. Spark writes to `/tmp` inside the container and the finished
files are copied onto the mount, so Dropbox no longer has to be paused for a
write on the order of a gigabyte. See `staged_write.py`.

In [1]:
import os

from bootstrap import MONGO_DB, NDJSON_DIR, mongo_db, start_spark
from schemas import FINANCIAL_SCHEMA
from staged_write import write_staged

# Paths, the session and the JVM readout all come from bootstrap.py, so the
# five notebooks that open a session document the environment identically.
# Driver memory, thread count, the Mongo connector package and the connection
# URI still come from jupyter/spark-defaults.conf, which is baked into the
# image. Change the config file and rebuild.
spark = start_spark("group13_ndjson_export")

db = mongo_db(spark)

spark_version        4.2.0
spark_master         local[4]
driver_max_heap_gb   8.0 GB
default_parallelism  4
cpu_cores            12
connector            org.mongodb.spark:mongo-spark-connector_2.13:11.1.0
mongo_uri            mongodb://mongodb:27017


## Change detection

Same signature approach as the Parquet export. A full rewrite takes minutes, so
it is skipped when the source is unchanged. Merge logic would add failure modes
to save a few minutes at this size.

In [2]:
import json
import os

from mirrors import load_metadata, save_metadata, source_signature, stale_collections

METADATA_PATH = os.path.join(NDJSON_DIR, "_export_metadata.json")

# Only financial_data is mirrored to NDJSON. The companies mirror exists in
# Parquet only; variant D reads the raw register file directly for that side,
# which is the comparison the benchmark is actually making.
COLLECTIONS = ["financial_data"]

# Set True to re-export even when the source signature is unchanged.
FORCE_REFRESH = False

previous = load_metadata(METADATA_PATH, COLLECTIONS)
current = source_signature(db, COLLECTIONS)
stale = stale_collections(current, previous["signature"], force=FORCE_REFRESH)["financial_data"]

print(json.dumps(current, indent=2))
print()
print("financial_data  %s" % ("export needed" if stale else "unchanged, skipping"))

{
  "financial_data": {
    "count": 1170290,
    "max_fetched_at": "2026-09-11T05:43:27.225000"
  }
}

financial_data  export needed


## Export

Read through the connector with the full `FINANCIAL_SCHEMA`, including the
nested `data` statement blob, then write as NDJSON.

The write is repartitioned to `defaultParallelism`. Without it the connector's
own partitioning propagates through, which produced an uneven file layout; with
it the reader gets one split per core. This is a deliberate advantage handed to
the NDJSON variant, and the benchmark notes it: the raw `companies` file gets
no such help, because it arrives as a single unsplittable array and nothing can
be done about that without rewriting it.

In [3]:
os.makedirs(NDJSON_DIR, exist_ok=True)
TARGET = os.path.join(NDJSON_DIR, "financial_data")

if stale:
    df = (
        spark.read.format("mongodb")
        .option("database", MONGO_DB)
        .option("collection", "financial_data")
        .schema(FINANCIAL_SCHEMA)
        .load()
    )
    # Staged through container-local disk, then copied onto the mount: Spark's
    # commit renames fail intermittently against Dropbox, and they fail after
    # mode("overwrite") has deleted the previous export. See staged_write.py.
    write_staged(df.repartition(spark.sparkContext.defaultParallelism),
                 TARGET, "json")
    # Count from the written files rather than the DataFrame, which would
    # otherwise re-read the whole collection from MongoDB a second time.
    rows = spark.read.schema(FINANCIAL_SCHEMA).json(TARGET).count()
    print("wrote %d rows" % rows)

    # Written only after a successful export, so an interrupted run stays
    # marked stale rather than falsely up to date.
    save_metadata(METADATA_PATH, current, {"financial_data": rows}, previous)
    print("Metadata updated.")
else:
    # Nothing written, but the mirror was checked and found current. Recording
    # that keeps the benchmark's staleness check able to tell an unchanged
    # mirror from one that was never exported. See mirrors.save_metadata.
    save_metadata(METADATA_PATH, current, {}, previous, wrote=False)
    print("Skipped; recorded as checked and current.")

  staged write: 10 files, 0.69 GB copied to /home/jovyan/data/ndjson/financial_data
wrote 1170290 rows
Metadata updated.


## Verification

Two things are checked. Row count against MongoDB, and a round-trip of the
statement values.

The round-trip matters because Spark serialises `fetched_at` as an ISO-8601
string in JSON and must parse it back as a timestamp. If that fails the column
silently becomes null, and the benchmark would compare a working Parquet read
against a broken NDJSON read.

In [4]:
from pyspark.sql import functions as F

back = spark.read.schema(FINANCIAL_SCHEMA).json(TARGET)

rows = back.count()
mongo_rows = db.financial_data.count_documents({})
print("rows  ndjson=%d  mongo=%d  %s"
      % (rows, mongo_rows, "OK" if rows == mongo_rows else "MISMATCH"))
assert rows == mongo_rows, "ndjson/mongo row mismatch"

# Nulls here would mean a parse failure, not missing source data.
checks = back.select(
    F.sum(F.col("fetched_at").isNull().cast("int")).alias("null_fetched_at"),
    F.sum(F.col("data").isNotNull().cast("int")).alias("with_statement"),
    F.sum(F.col("data")[0]["resultatregnskapResultat"]["aarsresultat"]
          .isNotNull().cast("int")).alias("with_aarsresultat"),
    F.sum(F.col("data")[0]["resultatregnskapResultat"]["totalresultat"]
          .isNotNull().cast("int")).alias("with_totalresultat"),
).collect()[0]

# Measured from MongoDB in the same run rather than compared against a stored
# figure. Mirrors Spark's isNotNull: missing and explicit null both count as null.
# "$ne: None" excludes both missing fields and explicit nulls, matching Spark's
# isNotNull. ("missing" is only a valid type alias in the aggregation $type
# expression, not in a query.)
mongo_with_data = db.financial_data.count_documents({"data": {"$ne": None}})

print("\n%-30s %12s %12s" % ("metric", "ndjson", "mongo"))
print("-" * 58)
print("%-30s %12d %12d %s" % ("records with data", checks["with_statement"],
                              mongo_with_data,
                              "OK" if checks["with_statement"] == mongo_with_data
                              else "MISMATCH"))
assert checks["with_statement"] == mongo_with_data, "ndjson/mongo data mismatch"

# A non-zero count here means a parse failure, which is true at any corpus size
# and needs no reference value.
print("\nnull fetched_at:      %d   (must be 0)" % checks["null_fetched_at"])
assert checks["null_fetched_at"] == 0, "fetched_at failed to parse"

# Reported only: array-element null handling is the least obviously equivalent
# part of the two engines, so a difference is information, not a failure.
print("with aarsresultat:    %d" % checks["with_aarsresultat"])
print("with totalresultat:   %d" % checks["with_totalresultat"])

size = sum(os.path.getsize(os.path.join(d, f))
           for d, _, files in os.walk(TARGET) for f in files)
print("\nNDJSON on disk: %.2f GB in %d files"
      % (size / 1024**3, len(os.listdir(TARGET))))

rows  ndjson=1170290  mongo=1170290  OK

metric                               ndjson        mongo
----------------------------------------------------------
records with data                    445359       445359 OK

null fetched_at:      0   (must be 0)
with aarsresultat:    445359
with totalresultat:   242047

NDJSON on disk: 0.69 GB in 10 files
